# Phase 3 GPU fine-tune (Colab or Kaggle) — wav2vec2-base + AASIST-style head

No-GPU rule: this repo's CPU cannot fine-tune encoders, so encoder fine-tuning happens HERE. Two paths:
(A) frozen-backbone + attentive head (works on free T4, ~30 min); (B) last-2-layers unfrozen (needs 16 GB VRAM).

## 1. What to UPLOAD to this runtime (from your repo checkout)
- `data/phase2_manifest.csv` (419 rows; commit `6b770ff`)
- `data/synth_vc/*.wav` (240 files) + the LibriSpeech clips it references — OR set `USE_PUBLIC=1` below to pull WaveFake/ASVspoof instead (needs the dataset agreements; L sets need your approval first)
- `models/stacking.json` (VAL-fit stacking, reused as-is — never refit here)

## 2. What to DOWNLOAD back to the repo
- `head_finetuned.pt` → `models/` (A), or `encoder_finetuned/` dir → `models/` (B)
- `finetune_report.json` → `reports/` (all numbers the UI will ever show)

`my_samples` (samples/labels.csv, 19 files) is NEVER uploaded here — it stays untouched until Phase 4.

In [ ]:
!pip install -q torch torchaudio transformers scikit-learn soundfile librosa
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only')

In [ ]:
# --- config ---
MANIFEST = 'phase2_manifest.csv'  # uploaded (see cell 0)
USE_PUBLIC = 0  # set 1 to ALSO pull WaveFake-small (S-M, no approval needed beyond dataset terms)
PATH_A_OUT, PATH_B_OUT = 'head_finetuned.pt', 'encoder_finetuned'
SEED = 7

import csv, random
rows = [r for r in csv.DictReader(open(MANIFEST, encoding='utf-8')) if r['split'] in ('train', 'val')]
print(len(rows), 'train+val rows;', sum(1 for r in rows if r['split'] == 'train'), 'train')
assert not any(r['split'] == 'my_samples' for r in rows), 'my_samples leaked into GPU manifest!'

In [ ]:
# --- codec augmentation, BOTH classes, train rows only (mirrors tools/augment_phase2.py) ---
import numpy as np, soundfile as sf, subprocess, tempfile, shutil
from pathlib import Path
!apt-get install -y -qq ffmpeg 2>&1 | tail -1

def codec_roundtrip(x16, codec):
    tmp = Path(tempfile.mkdtemp()); src = tmp / 's.wav'; sf.write(str(src), x16, 16000)
    if codec == 'opus': enc, cmd = tmp/'o.opus', ['ffmpeg','-y','-v','error','-i',str(src),'-c:a','libopus','-b:a','24k',str(tmp/'o.opus')]
    elif codec == 'mp3': enc, cmd = tmp/'o.mp3', ['ffmpeg','-y','-v','error','-i',str(src),'-c:a','libmp3lame','-b:a','128k',str(tmp/'o.mp3')]
    elif codec == 'amr': enc, cmd = tmp/'o.amr', ['ffmpeg','-y','-v','error','-i',str(src),'-ar','8000','-ac','1','-c:a','amr_nb',str(tmp/'o.amr')]
    else: enc, cmd = tmp/'o.wav', ['ffmpeg','-y','-v','error','-i',str(src),'-ar','8000','-ac','1','-c:a','pcm_alaw',str(tmp/'o.wav')]
    subprocess.run(cmd, check=True, timeout=120)
    back = tmp/'b.wav'
    subprocess.run(['ffmpeg','-y','-v','error','-i',str(enc),'-ar','16000','-ac','1',str(back)], check=True, timeout=120)
    a, _ = sf.read(str(back), dtype='float32'); shutil.rmtree(tmp, ignore_errors=True)
    return np.asarray(a, dtype=np.float32).ravel()

print('ffmpeg ok:', subprocess.run(['ffmpeg','-version'], capture_output=True, text=True).stdout.splitlines()[0][:80])

In [ ]:
# --- Path A: frozen wav2vec2-base + attentive-pooling head (T4-friendly) ---
import torch, torch.nn as nn
from transformers import AutoFeatureExtractor, AutoModel
BB = 'facebook/wav2vec2-base'
feat = AutoFeatureExtractor.from_pretrained(BB)
bb = AutoModel.from_pretrained(BB).eval()
for p in bb.parameters(): p.requires_grad = False

class AttentiveHead(nn.Module):
    def __init__(self, d=768, h=256):
        super().__init__()
        self.wsum = nn.Parameter(torch.ones(13) / 13)  # layer-weighted sum incl. embeddings
        self.att = nn.Linear(d, 1); self.mlp = nn.Sequential(nn.Linear(d, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.2), nn.Linear(h, 1))
    def forward(self, hidden):  # list of 13 [B,T,D]
        w = torch.softmax(self.wsum, 0)
        h = sum(wi * hi for wi, hi in zip(w, hidden))
        a = torch.softmax(self.att(h).squeeze(-1), -1)  # attentive pooling over time
        return self.mlp((a.unsqueeze(-1) * h).sum(1)).squeeze(-1)

head = AttentiveHead(); print(sum(p.numel() for p in head.parameters())/1e3, 'k trainable')

In [ ]:
# --- train head on train split (with codec aug on BOTH classes), early-stop on val macro-F1 ---
import csv, random
import numpy as np, soundfile as sf, torch
from sklearn.metrics import f1_score
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; bb.to(dev); head.to(dev)
opt = torch.optim.AdamW(head.parameters(), lr=3e-4); ce = torch.nn.BCEWithLogitsLoss()

def load16(path, cap=6):
    a, sr = sf.read(path, dtype='float32'); a = np.asarray(a, dtype=np.float32).ravel()
    if sr != 16000: a = np.interp(np.linspace(0, 1, int(len(a)/sr*16000)), np.linspace(0, 1, len(a)), a).astype(np.float32)
    return a[:16000*cap]

tr = [r for r in rows if r['split'] == 'train']; va = [r for r in rows if r['split'] == 'val']
AUG = [None, 'opus', 'mp3', 'amr']
best, wait, best_state = -1, 0, None
for ep in range(30):
    head.train(); random.shuffle(tr)
    for r in tr:
        x = load16(r['path']); c = random.choice(AUG)
        if c: x = codec_roundtrip(x, c)  # label-blind: same AUG distribution both classes
        inp = feat(x, sampling_rate=16000, return_tensors='pt')
        with torch.inference_mode(): h = bb(**{k: v.to(dev) for k, v in inp.items() if k in ('input_values','attention_mask')}, output_hidden_states=True).hidden_states
        logit = head([hi.to(dev) for hi in h])
        y = torch.tensor([0.0 if r['label_origin'] == 'HUMAN_LIVE' else 1.0], device=dev)
        opt.zero_grad(); loss = ce(logit.unsqueeze(0) if logit.ndim == 0 else logit, y); loss.backward(); opt.step()
    head.eval(); ps, ys = [], []
    with torch.inference_mode():
        for r in va:
            x = load16(r['path']); inp = feat(x, sampling_rate=16000, return_tensors='pt')
            h = bb(**{k: v.to(dev) for k, v in inp.items() if k in ('input_values','attention_mask')}, output_hidden_states=True).hidden_states
            ps.append(float(torch.sigmoid(head([hi.to(dev) for hi in h])))); ys.append(0 if r['label_origin'] == 'HUMAN_LIVE' else 1)
    f1 = f1_score(ys, [1 if p >= 0.5 else 0 for p in ps], average='macro')
    print(f'ep{ep} val macro-F1 {f1:.4f}')
    if f1 > best + 1e-4: best, wait, best_state = f1, 0, {k: v.cpu() for k, v in head.state_dict().items()}
    else:
        wait += 1
        if wait >= 6: break
head.load_state_dict(best_state); print('best val macro-F1:', best)

In [ ]:
# --- save + report (download these two files back to the repo) ---
import json
torch.save({'head': head.state_dict(), 'backbone': BB, 'seed': SEED}, 'head_finetuned.pt')
rep = {'val_macro_f1': best, 'seed': SEED, 'backbone': BB, 'path': 'A-frozen-head',
       'note': 'VAL only; TEST/my_samples untouched. Copy head_finetuned.pt -> repo models/ and report -> reports/finetune_report.json'}
json.dump(rep, open('finetune_report.json', 'w'), indent=2); print(rep)
print('DOWNLOAD: head_finetuned.pt + finetune_report.json')
# Path B (optional, 16 GB VRAM): unfreeze bb.encoder.layers[-2:], lr 1e-5, same loop, save encoder_finetuned/